# Comparaison de 2 baselines vs dernier run

Ce notebook recalcule les métriques **directement depuis les fichiers `results.csv`** de tes runs d'évaluation.

Il permet de comparer :
- baseline historique vs dernier run
- baseline uploadée / officielle vs dernier run

L'idée est de **ne pas dépendre d'un calcul manuel fait en discussion**.


## Mode d'emploi

1. Renseigne les chemins vers les trois fichiers `results.csv`.
2. Exécute toutes les cellules.
3. Lis les deux tableaux de comparaison générés automatiquement.

Tu peux mettre soit des chemins absolus Windows, soit des chemins relatifs au projet.


In [1]:
from pathlib import Path

# --- À MODIFIER ---
# Exemple Windows : r"C:\\Users\\thoma\\Documents\\Openclassroom\\Projet-10\\evals\\experiments\\run_20260313_192855\\results.csv"

BASELINE_1_RESULTS = r"C:\\Users\\thoma\\Documents\\Openclassroom\\Projet-10\\evals\\experiments\\run_20260310_174748\\results.csv"
BASELINE_2_RESULTS = r"C:\\Users\\thoma\\Documents\\Openclassroom\\Projet-10\\evals\\experiments\\run_20260313_160107\\results.csv"
FINAL_RESULTS      = r"C:\\Users\\thoma\\Documents\\Openclassroom\\Projet-10\\evals\\experiments\\run_20260313_192855\\results.csv"

# Noms affichés dans les tableaux
BASELINE_1_NAME = "Baseline historique"
BASELINE_2_NAME = "Baseline uploadée"
FINAL_NAME      = "Dernier run"

assert BASELINE_1_RESULTS, "Renseigne BASELINE_1_RESULTS"
assert BASELINE_2_RESULTS, "Renseigne BASELINE_2_RESULTS"
assert FINAL_RESULTS, "Renseigne FINAL_RESULTS"

for p in [BASELINE_1_RESULTS, BASELINE_2_RESULTS, FINAL_RESULTS]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Fichier introuvable : {p}")

print('Chemins OK.')


Chemins OK.


In [2]:
import pandas as pd
import numpy as np

KEY_METRICS = [
    'faithfulness',
    'answer_relevancy',
    'hit_at_k',
    'mrr',
    'recall_at_k',
    'precision_at_k',
    'tool_routing_ok',
    'tool_call_accuracy_ragas',
    'tool_call_precision_ragas',
    'tool_call_recall_ragas',
    'tool_call_f1_ragas',
    'sql_row_count',
]

def load_results_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df

def numeric_means(df: pd.DataFrame) -> pd.Series:
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.empty:
        return pd.Series(dtype='float64')
    return numeric_df.mean(numeric_only=True)

def build_metric_table(df: pd.DataFrame, run_name: str) -> pd.DataFrame:
    means = numeric_means(df)
    rows = []
    for metric in KEY_METRICS:
        value = means.get(metric, np.nan)
        rows.append({'metric': metric, run_name: value})
    return pd.DataFrame(rows)

def safe_pct_delta(base, final):
    if pd.isna(base) or pd.isna(final):
        return np.nan
    if base == 0:
        return np.nan
    return ((final - base) / abs(base)) * 100

def compare_runs(base_df: pd.DataFrame, final_df: pd.DataFrame, base_name: str, final_name: str) -> pd.DataFrame:
    base_means = numeric_means(base_df)
    final_means = numeric_means(final_df)
    rows = []
    for metric in KEY_METRICS:
        base_value = base_means.get(metric, np.nan)
        final_value = final_means.get(metric, np.nan)
        abs_delta = final_value - base_value if pd.notna(base_value) and pd.notna(final_value) else np.nan
        pct_delta = safe_pct_delta(base_value, final_value)
        rows.append({
            'metric': metric,
            base_name: base_value,
            final_name: final_value,
            'delta_absolu': abs_delta,
            'delta_%': pct_delta,
        })
    comp = pd.DataFrame(rows)
    return comp

def format_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    numeric_cols = [c for c in out.columns if c != 'metric']
    for col in numeric_cols:
        out[col] = out[col].map(lambda x: round(x, 6) if pd.notna(x) else x)
    return out


In [3]:
baseline1_df = load_results_csv(BASELINE_1_RESULTS)
baseline2_df = load_results_csv(BASELINE_2_RESULTS)
final_df = load_results_csv(FINAL_RESULTS)

print(BASELINE_1_NAME, baseline1_df.shape)
print(BASELINE_2_NAME, baseline2_df.shape)
print(FINAL_NAME, final_df.shape)


Baseline historique (50, 23)
Baseline uploadée (50, 36)
Dernier run (50, 36)


## Moyennes numériques par run

Cette section permet de voir les métriques moyennes calculées directement depuis chaque `results.csv`.


In [4]:
summary_table = build_metric_table(baseline1_df, BASELINE_1_NAME) \
    .merge(build_metric_table(baseline2_df, BASELINE_2_NAME), on='metric', how='outer') \
    .merge(build_metric_table(final_df, FINAL_NAME), on='metric', how='outer')

format_table(summary_table)


,metric,Baseline historique,Baseline uploadée,Dernier run
0,answer_relevancy,0.518936,0.824918,0.817038
1,faithfulness,0.530251,0.930548,0.954032
2,hit_at_k,0.444444,0.400000,0.422222
3,mrr,0.444444,0.388889,0.422222
4,precision_at_k,0.088889,0.080000,0.084444
5,recall_at_k,0.344444,0.322222,0.344444
6,sql_row_count,NaN,1.972222,1.945946
7,tool_call_accuracy_ragas,NaN,0.955556,0.955556
8,tool_call_f1_ragas,NaN,0.985185,0.985185
9,tool_call_precision_ragas,NaN,1.000000,1.000000


## Comparaison 1 : baseline historique vs dernier run


In [5]:
comp_1 = compare_runs(baseline1_df, final_df, BASELINE_1_NAME, FINAL_NAME)
format_table(comp_1)


,metric,Baseline historique,Dernier run,delta_absolu,delta_%
0,faithfulness,0.530251,0.954032,0.423781,79.920766
1,answer_relevancy,0.518936,0.817038,0.298102,57.444796
2,hit_at_k,0.444444,0.422222,-0.022222,-5.000000
3,mrr,0.444444,0.422222,-0.022222,-5.000000
4,recall_at_k,0.344444,0.344444,0.000000,0.000000
5,precision_at_k,0.088889,0.084444,-0.004444,-5.000000
6,tool_routing_ok,NaN,0.955556,NaN,NaN
7,tool_call_accuracy_ragas,NaN,0.955556,NaN,NaN
8,tool_call_precision_ragas,NaN,1.000000,NaN,NaN
9,tool_call_recall_ragas,NaN,0.977778,NaN,NaN


## Comparaison 2 : baseline uploadée vs dernier run


In [6]:
comp_2 = compare_runs(baseline2_df, final_df, BASELINE_2_NAME, FINAL_NAME)
format_table(comp_2)


,metric,Baseline uploadée,Dernier run,delta_absolu,delta_%
0,faithfulness,0.930548,0.954032,0.023484,2.523689
1,answer_relevancy,0.824918,0.817038,-0.007879,-0.955167
2,hit_at_k,0.400000,0.422222,0.022222,5.555556
3,mrr,0.388889,0.422222,0.033333,8.571429
4,recall_at_k,0.322222,0.344444,0.022222,6.896552
5,precision_at_k,0.080000,0.084444,0.004444,5.555556
6,tool_routing_ok,0.955556,0.955556,0.000000,0.000000
7,tool_call_accuracy_ragas,0.955556,0.955556,0.000000,0.000000
8,tool_call_precision_ragas,1.000000,1.000000,0.000000,0.000000
9,tool_call_recall_ragas,0.977778,0.977778,0.000000,0.000000


## Lecture rapide des écarts

- `delta_absolu` > 0 : le dernier run est plus élevé
- `delta_absolu` < 0 : le dernier run est plus faible
- `delta_%` : variation relative, calculée par rapport au baseline

Attention : selon la métrique, une hausse n'est pas toujours une amélioration métier. Par exemple `sql_row_count` plus faible peut être souhaitable si les requêtes sont plus ciblées.


In [7]:
# Export optionnel des tableaux
EXPORT_DIR = Path('comparison_exports')
EXPORT_DIR.mkdir(exist_ok=True)

format_table(summary_table).to_csv(EXPORT_DIR / 'all_runs_summary.csv', index=False)
format_table(comp_1).to_csv(EXPORT_DIR / 'baseline1_vs_final.csv', index=False)
format_table(comp_2).to_csv(EXPORT_DIR / 'baseline2_vs_final.csv', index=False)

print('Exports créés dans :', EXPORT_DIR.resolve())


Exports créés dans : C:\Users\thoma\Documents\Openclassroom\Projet-10\notebooks\comparison_exports
